# 이미지 화질 개선 고급 스킬 - 실사용 사진 전처리

앞서 화질 진단에서 확인한 것처럼, 학습 데이터(AI-Hub 스튜디오 사진)는 전부 깨끗해서
모델이 흔들림/저조도/압축 같은 실사용 조건을 한 번도 학습 중에 본 적이 없다. 이 노트북은
그 격차를 줄이기 위한 **추론 시점 이미지 개선 파이프라인**을 만들고, 실제 사진으로
얼마나 회복되는지 수치와 눈으로 직접 검증한다.

적용 순서: 화이트밸런스 보정 -> 노이즈 제거 -> 저조도 보정(감마+CLAHE) -> 샤프닝
(순서가 중요하다 - 노이즈를 먼저 지우지 않고 샤프닝부터 하면 노이즈까지 같이 증폭된다.
실제로 처음 만들었을 때 이 실수를 했다가 결과가 더 나빠지는 걸 확인하고 순서를 바꿨다.)

In [ ]:
#@title (1) Import + 데이터 경로
import os
import json
import random

import numpy as np
import cv2
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

DATA_ROOT = "/Users/codeit/Downloads/sprint_ai_project1_data"
TRAIN_IMAGE_DIR = os.path.join(DATA_ROOT, "train_images")

with open(os.path.join(DATA_ROOT, "tightened_annotations.json"), "r", encoding="utf-8") as f:
    image_records = json.load(f)

print(f"이미지 {len(image_records)}장 로드 완료")


## STEP 1 : 실사용 환경 시뮬레이션

학습 데이터는 다 깨끗하니, 대표적인 "나쁜 사용자 사진" 조건 4가지를 인위적으로 합성해서
비교 기준으로 삼는다.

In [ ]:
#@title (1-A) 열화 시뮬레이션 함수
def simulate_motion_blur(img, k=15):
    kernel = np.zeros((k, k))
    kernel[k // 2, :] = 1.0 / k
    return cv2.filter2D(img, -1, kernel)


def simulate_low_light(img, factor=0.35):
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)


def simulate_noise(img, sigma=20):
    noise = np.random.normal(0, sigma, img.shape)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def simulate_jpeg_compression(img, quality=15):
    ok, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)


def simulate_bad_photo(img):
    """손떨림 + 어두운 방 + 저가 카메라 노이즈 + 메신저로 전송하면서 압축까지,
    최악을 가정한 실사용 사진 하나로 합쳐서 재현."""
    return simulate_jpeg_compression(simulate_noise(simulate_low_light(simulate_motion_blur(img))))


## STEP 2 : 화질 개선 파이프라인

색(화이트밸런스) -> 노이즈 -> 밝기 -> 선명도 순서로 처리한다. 각 단계는 원본이 이미
괜찮으면 최소한으로만 개입하도록(저조도가 아니면 감마 보정 자체를 건너뜀 등) 조건을 뒀다
- 이미 깨끗한 사진을 괜히 더 건드려서 오히려 나빠지는 걸 막기 위함.

In [ ]:
#@title (2-A) 화이트밸런스 보정
# 색이 알약 판별의 핵심 단서인데, 카메라/조명마다 화이트밸런스가 달라서 같은 약도
# 색이 다르게 찍힐 수 있다. Gray-World 가정(전체 평균은 무채색이어야 한다)으로
# LAB 색공간의 a/b 채널을 밝기에 비례해서 보정한다.
def auto_white_balance(bgr):
    result = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    avg_a = np.average(result[:, :, 1])
    avg_b = np.average(result[:, :, 2])
    result[:, :, 1] = np.clip(result[:, :, 1] - ((avg_a - 128) * (result[:, :, 0] / 255.0) * 1.1), 0, 255)
    result[:, :, 2] = np.clip(result[:, :, 2] - ((avg_b - 128) * (result[:, :, 0] / 255.0) * 1.1), 0, 255)
    return cv2.cvtColor(result.astype(np.uint8), cv2.COLOR_LAB2BGR)


In [ ]:
#@title (2-B) 저조도 보정 (감마 보정 + CLAHE)
# 목표 밝기(TARGET_BRIGHTNESS)로 감마 보정한 뒤, L채널에만 CLAHE를 걸어서 국소
# 대비도 같이 살린다(색은 안 건드림 - a/b 채널은 그대로 둬서 색이 틀어지지 않게 함).
TARGET_BRIGHTNESS = 130.0


def gamma_correct(bgr, gamma):
    table = ((np.arange(256) / 255.0) ** (1.0 / gamma) * 255).astype(np.uint8)
    return cv2.LUT(bgr, table)


def fix_low_light(bgr):
    brightness = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).mean()
    if brightness >= 120:
        return bgr  # 이미 충분히 밝으면 손대지 않음

    # brightness^(1/gamma) = TARGET 이 되도록 gamma 역산.
    # gamma > 1일 때 어두운 값이 더 크게 밝아진다 (감마 보정의 표준 방향).
    gamma = float(np.clip(
        np.log(max(brightness, 1) / 255.0) / np.log(TARGET_BRIGHTNESS / 255.0), 1.0, 6.0
    ))
    out = gamma_correct(bgr, gamma)

    lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)


In [ ]:
#@title (2-C) 전체 파이프라인 (화이트밸런스 -> 디노이즈 -> 저조도 보정 -> 샤프닝)
def enhance_image(bgr):
    out = auto_white_balance(bgr)

    # 디노이즈를 먼저: 나중에 샤프닝할 때 노이즈까지 같이 증폭되는 걸 막기 위함
    out = cv2.fastNlMeansDenoisingColored(out, None, h=12, hColor=12, templateWindowSize=7, searchWindowSize=21)

    out = fix_low_light(out)

    # 언샤프 마스킹으로 완만하게 선명도 회복 (디노이즈 이후라 노이즈 증폭 걱정이 적음)
    gaussian = cv2.GaussianBlur(out, (0, 0), sigmaX=2)
    out = cv2.addWeighted(out, 1.4, gaussian, -0.4, 0)

    return out


## STEP 3 : 실제 데이터로 검증

랜덤 20장을 골라서 "원본 -> 인위적 열화 -> 화질 개선"을 거친 뒤, 밝기가 원본에 얼마나
다시 가까워지는지 정량적으로 확인하고, 대표 이미지 몇 장은 눈으로도 확인한다.

In [ ]:
#@title (3-A) 정량 검증 - 밝기/선명도 회복률
def blur_score(bgr):
    return cv2.Laplacian(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()


def brightness_of(bgr):
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).mean()


sample_stems = random.sample(list(image_records.keys()), 20)
rows = []
for stem in sample_stems:
    img = cv2.imread(os.path.join(TRAIN_IMAGE_DIR, image_records[stem]["file_name"]))
    bad = simulate_bad_photo(img)
    fixed = enhance_image(bad)
    rows.append((brightness_of(img), brightness_of(bad), brightness_of(fixed)))

bo, bd, be = zip(*rows)
gap_before = np.mean([abs(o - d) for o, d in zip(bo, bd)])
gap_after = np.mean([abs(o - e) for o, e in zip(bo, be)])

print(f"밝기 평균 - 원본 {np.mean(bo):.1f} | 열화 후 {np.mean(bd):.1f} | 개선 후 {np.mean(be):.1f}")
print(f"원본과의 밝기 격차: 열화 {gap_before:.1f} -> 개선 후 {gap_after:.1f}")
print(f"밝기 격차 회복률: {(1 - gap_after / gap_before):.1%}")


In [ ]:
#@title (3-B) 육안 확인 - 원본 / 열화 / 개선 나란히 비교
def show_enhance_comparison(stems, n=4):
    fig, axes = plt.subplots(len(stems[:n]), 3, figsize=(9, 3 * len(stems[:n])))
    axes = np.atleast_2d(axes)

    for row, stem in enumerate(stems[:n]):
        img = cv2.imread(os.path.join(TRAIN_IMAGE_DIR, image_records[stem]["file_name"]))
        bad = simulate_bad_photo(img)
        fixed = enhance_image(bad)

        for col, (title, im) in enumerate([("원본", img), ("실사용 가정(열화)", bad), ("화질 개선 후", fixed)]):
            axes[row, col].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            axes[row, col].set_title(title, fontsize=9)
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()


show_enhance_comparison(sample_stems, n=4)


In [ ]:
#@title (3-C) wandb에 화질 개선 검증 결과 기록
import wandb

run = wandb.init(
    entity="jaedong0817--org",
    project="beginner-team-project",
    job_type="preprocess",
    name="image-quality-enhancement",
    config={
        "target_brightness": TARGET_BRIGHTNESS,
        "denoise_h": 12,
        "sharpen_amount": 1.4,
        "n_test_samples": len(sample_stems),
    },
)

run.summary["brightness_original_mean"] = float(np.mean(bo))
run.summary["brightness_degraded_mean"] = float(np.mean(bd))
run.summary["brightness_enhanced_mean"] = float(np.mean(be))
run.summary["brightness_gap_before"] = float(gap_before)
run.summary["brightness_gap_after"] = float(gap_after)
run.summary["brightness_recovery_rate"] = float(1 - gap_after / gap_before)

run.finish()
print("wandb 기록 완료. project beginner-team-project에서 image-quality-enhancement run으로 확인 가능.")


## STEP 4 : 추론 파이프라인에 실제로 연결하기

학습 노트북(RetinaNet_ResNet50_FPN_v2_AIHub_Merged_클래스밸런스.ipynb)의 TestDataset은
지금 원본 이미지를 그대로 읽어서 모델에 넣는다. 실제 사용자 사진은 화질이 나쁠 수 있으니,
읽은 직후 `enhance_image()`를 한 번 거치도록 아래처럼 바꿔서 쓰면 된다.

```python
# TestDataset.__getitem__ 안에서, 이미지를 읽은 직후에 추가
with Image.open(os.path.join(self.image_dir, file_name)) as im:
    image = im.convert("RGB")

image_np = enhance_image(np.array(image))  # <- 이 줄만 추가
image = Image.fromarray(image_np)
```

주의할 점: **학습 데이터에는 이 보정을 걸면 안 된다.** 학습은 원본 그대로 두고
(대신 STEP 1의 열화 시뮬레이션을 augmentation으로 추가하는 걸 권장 - 모델이 나쁜
화질에 자체적으로 강해지도록), 실제 테스트/서비스 입력에만 이 보정을 거는 게 맞다.
두 개를 섞으면 train/test 분포가 오히려 어긋난다.